# Indexation booléenne : garder les extraits et leurs cibles

Décomposer une sélection au lieu de cacher trois opérations dans une seule ligne.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#filtrer-et-preserver-les-cibles).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Reprenons les trois extraits fictifs. Nous voulons garder ceux dont le facteur de crête dépasse 2. Il faut conserver leurs annotations correspondantes.

| Extrait | Facteur de crête | Dépasse 2 ? | Cible si gardé |
| --- | --- | --- | --- |
| A | 1,5 | False | — |
| B | 4,5 | True | 1 |
| C | 2,5 | True | 0 |

Le masque concerne l’axe des extraits. Il doit sélectionner les mêmes positions dans X et dans cibles.

## Démonstration 1 · Séparer colonne, comparaison et filtrage

Lire chaque opération avant de les raccourcir.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [4]:
X = torch.tensor([[0.1, 1.5], [0.2, 4.5], [0.15, 2.5]])
cibles = torch.tensor([0, 1, 0], dtype=torch.long)


In [5]:
observer('X', X)
observer('cibles', cibles)

X forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[0.10000000149011612, 1.5], [0.20000000298023224, 4.5], [0.15000000596046448, 2.5]]
cibles forme (3,) type torch.int64 requires_grad False
  valeurs : [0, 1, 0]


### Calculer l’état suivant

X[:,1] garde toutes les lignes avec :, puis choisit la colonne un : le facteur de crête.

In [6]:
cretes = X[:, 1]


In [7]:
observer('cretes', cretes)

cretes forme (3,) type torch.float32 requires_grad False
  valeurs : [1.5, 4.5, 2.5]


### Calculer l’état suivant

> 2.0 ne filtre pas encore X. Il fabrique les trois booléens du masque.

In [8]:
masque = cretes > 2.0


In [9]:
observer('masque', masque)

masque forme (3,) type torch.bool requires_grad False
  valeurs : [False, True, True]


### Calculer l’état suivant

X[masque] garde B et C en entier. Le tableau résultat a deux lignes et deux colonnes.

In [10]:
retenus = X[masque]


In [11]:
observer('retenus', retenus)

retenus forme (2, 2) type torch.float32 requires_grad False
  valeurs : [[0.20000000298023224, 4.5], [0.15000000596046448, 2.5]]


### Calculer l’état suivant

cibles[masque] garde les cibles de B puis C. Leur ordre reste celui des lignes retenues.

Cette expression compacte sélectionne la colonne, construit le masque puis choisit les lignes. Elle correspond aux trois premières instructions séparées.

In [12]:
cibles_retenues = cibles[masque]


In [13]:
observer('cibles_retenues', cibles_retenues)

cibles_retenues forme (2,) type torch.int64 requires_grad False
  valeurs : [1, 0]


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [14]:
print("cretes", cretes.tolist(), tuple(cretes.shape))
print("masque", masque.tolist(), tuple(masque.shape))
print("retenus", retenus.tolist(), tuple(retenus.shape))
print("cibles correspondantes", cibles_retenues.tolist())

cretes [1.5, 4.5, 2.5] (3,)
masque [False, True, True] (3,)
retenus [[0.20000000298023224, 4.5], [0.15000000596046448, 2.5]] (2, 2)
cibles correspondantes [1, 0]


Nous obtenons deux observations et exactement leurs deux cibles. Filtrer seulement les entrées casserait cette correspondance.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Prévoyez le masque qui conserve les lignes dont la seconde colonne dépasse 3.

In [15]:
exo_x = torch.tensor([[1.,2.],[3.,4.],[5.,6.]])
exo_masque = ...  # Remplacer les points par votre réponse.
if exo_masque is Ellipsis or exo_masque is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert torch.equal(torch.as_tensor(exo_masque),exo_x[:,1]>3)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

Le test utilise la seconde valeur de chaque ligne, pas les numéros de lignes.

</details>

### Exercice 2

Appliquez le même masque aux cibles. Quel vecteur accompagne les deux lignes retenues ?

In [16]:
exo_cibles = torch.tensor([0,1,0])
exo_masque = torch.tensor([False,True,True])
exo_cibles_gardees = ...  # Remplacer les points par votre réponse.
if exo_cibles_gardees is Ellipsis or exo_cibles_gardees is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert torch.equal(torch.as_tensor(exo_cibles_gardees),exo_cibles[exo_masque])
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Les positions 1 et 2 gardent leurs annotations dans cet ordre.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_x = torch.tensor([[1.,2.],[3.,4.],[5.,6.]])
exo_masque = torch.tensor([False,True,True])
if exo_masque is Ellipsis or exo_masque is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert torch.equal(torch.as_tensor(exo_masque),exo_x[:,1]>3)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_cibles = torch.tensor([0,1,0])
exo_masque = torch.tensor([False,True,True])
exo_cibles_gardees = torch.tensor([1,0])
if exo_cibles_gardees is Ellipsis or exo_cibles_gardees is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert torch.equal(torch.as_tensor(exo_cibles_gardees),exo_cibles[exo_masque])
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [17]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()